## TF-IDF

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE


In [ ]:
df = pd.read_csv('/content/mydata.csv', encoding='utf-8-sig')

In [ ]:
len(df)

In [ ]:
df.head(10)

In [ ]:
# df["label"] = df["label"].astype(int)

In [ ]:
df_neural = pd.read_csv("/content/myneural.csv", encoding="utf-8-sig")

In [ ]:
df_human = pd.read_csv("/content/myhuman.csv", encoding="utf-8-sig")

In [ ]:
df_train_val, df_test = train_test_split(
    df,
    test_size=0.20,
    stratify=df['label'],
    random_state=42
)

In [ ]:
print(f"Всего: {len(df)}, Тестовая: {len(df_test)}, Остаток (train+val): {len(df_train_val)}")

In [ ]:
df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.10,
    stratify=df_train_val['label'],
    random_state=42
)

In [ ]:
print(f"Обучающая: {len(df_train)}, Валидационная: {len(df_val)}")

In [ ]:
df_train.to_csv('/content/train.csv', index=False, encoding='utf-8-sig')
df_val.to_csv  ('/content/val.csv',   index=False, encoding='utf-8-sig')
df_test.to_csv ('/content/test.csv',  index=False, encoding='utf-8-sig')

In [ ]:
X_train = df_train['clean_text'].str.lower().values
X_val   = df_val  ['clean_text'].str.lower().values
X_test  = df_test ['clean_text'].str.lower().values

y_train = df_train['label'].values
y_val   = df_val  ['label'].values
y_test  = df_test ['label'].values

### **TF-IDF**

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w+\b'
)

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train)

In [ ]:
X_val_tfidf  = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
print('Train TF-IDF shape:', X_train_tfidf.shape)
print('Val   TF-IDF shape:', X_val_tfidf.shape)
print('Test  TF-IDF shape:', X_test_tfidf.shape)

In [ ]:
joblib.dump(vectorizer, '/content/tfidf_vectorizer.pkl')

## Логистическая регрессия

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

### Обучение

In [ ]:
model.fit(X_train_tfidf, y_train)

In [ ]:
joblib.dump(model, '/content/logreg_balanced.pkl')

### Валидация

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
y_val_pred = model.predict(X_val_tfidf)

print("=== Classification Report (валидация) ===")
print(classification_report(y_val, y_val_pred, target_names=['human', 'AI']))
print("=== Confusion Matrix ===")
print(confusion_matrix(y_val, y_val_pred))

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        stop_words='english',
        strip_accents='unicode',
        token_pattern=r'(?u)\b\w+\b'
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

In [ ]:
param_grid = {
    # сила регуляризации: меньшие C → больше регуляризации
    'clf__C':        [0.01, 0.1, 1, 10],
    # минимальная частота термов (количество документов)
    'tfidf__min_df': [1, 5, 10],
    # максимальная доля документов, в которых может встречаться термин
    'tfidf__max_df': [0.75, 0.85, 0.95]
}

In [ ]:
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
print("=== Лучшие параметры ===")
print(grid.best_params_)

In [ ]:
best_model = grid.best_estimator_
y_val_pred_grid = best_model.predict(X_val)

In [ ]:
print("=== Classification Report после GridSearch ===")
print(classification_report(y_val, y_val_pred_grid, target_names=['human', 'AI']))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

In [ ]:
y_pred = model.predict(X_test_tfidf)

In [ ]:
# матрица ошибок
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:")
print(cm)
print(f"True Negatives (human→human): {tn}")
print(f"False Positives (human→AI):   {fp}")
print(f"False Negatives (AI→human):    {fn}")
print(f"True Positives (AI→AI):        {tp}\n")

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['human', 'AI']))

In [ ]:
prec  = precision_score(y_test, y_pred, pos_label=1)
rec   = recall_score(y_test, y_pred, pos_label=1)
f1    = f1_score(y_test, y_pred, pos_label=1)
print(f"AI class — Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}\n")

In [ ]:
probs = model.predict_proba(X_test_tfidf)[:, 1]
threshold = 0.6  # повысить порог для большей точности
y_pred_thresh = (probs >= threshold).astype(int)
prec_t = precision_score(y_test, y_pred_thresh, pos_label=1)
rec_t  = recall_score(y_test, y_pred_thresh, pos_label=1)
f1_t   = f1_score(y_test, y_pred_thresh, pos_label=1)
print(f"Threshold {threshold} — Precision: {prec_t:.3f}, Recall: {rec_t:.3f}, F1: {f1_t:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve
)

In [ ]:
def plot_conf_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Human", "AI"])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(title)
    plt.show()

def plot_metrics_set(X, y_true, name, model):
    y_pred = model.predict(X)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    print(f"{name} — Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")
    plot_conf_matrix(y_true, y_pred, f"{name} — Confusion Matrix")

In [ ]:
# Визуализация метрик
plot_metrics_set(X_train_tfidf, y_train, "Train", model)
plot_metrics_set(X_val_tfidf, y_val, "Validation", model)
plot_metrics_set(X_test_tfidf, y_test, "Test", model)

In [ ]:
# ROC-кривая
fpr, tpr, thresholds = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC-кривая (тест)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Precision-Recall кривая
precision, recall, thresholds = precision_recall_curve(y_test, probs)

plt.figure()
plt.plot(recall, precision, label='Precision-Recall curve')
plt.title('PR-кривая (тест)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_score, recall_score, f1_score, classification_report
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve

In [ ]:
# 1. Загрузим датасеты и приведём метки 0→'human', 1→'AI'
df = pd.read_csv('/content/mydata.csv', encoding='utf-8-sig')
label_mapping = {0: 'human', 1: 'AI'}
df['label'] = df['label'].map(label_mapping)

# (Доп. данные, если нужны для чего-то ещё; оставлено без изменений)
# df_neural = pd.read_csv("/content/myneural.csv", encoding="utf-8-sig")
# df_human  = pd.read_csv("/content/myhuman.csv",  encoding="utf-8-sig")

In [ ]:
# 2. Разбиваем на train+val и test
df_train_val, df_test = train_test_split(
    df,
    test_size=0.20,
    stratify=df['label'],
    random_state=42
)
print(f"Всего: {len(df)}, Тестовая: {len(df_test)}, Остаток (train+val): {len(df_train_val)}")

# 3. Разбиваем train+val на train и val
df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.10,
    stratify=df_train_val['label'],
    random_state=42
)
print(f"Обучающая: {len(df_train)}, Валидационная: {len(df_val)}")

# Сохраним CSV (если нужно)
df_train.to_csv('/content/train.csv', index=False, encoding='utf-8-sig')
df_val.to_csv(  '/content/val.csv',   index=False, encoding='utf-8-sig')
df_test.to_csv( '/content/test.csv',  index=False, encoding='utf-8-sig')

# 4. Подготовим X и y (нормализуем текст в нижний регистр)
X_train = df_train['clean_text'].str.lower().values
X_val   = df_val[  'clean_text'].str.lower().values
X_test  = df_test[ 'clean_text'].str.lower().values

y_train = df_train['label'].values
y_val   = df_val[  'label'].values
y_test  = df_test[ 'label'].values

In [ ]:
# 5. TF-IDF векторизация
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w+\b'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

print('Train TF-IDF shape:', X_train_tfidf.shape)
print('Val   TF-IDF shape:', X_val_tfidf.shape)
print('Test  TF-IDF shape:', X_test_tfidf.shape)

# Сохраним векторизатор
joblib.dump(vectorizer, '/content/tfidf_vectorizer.pkl')

In [ ]:
# 6. Обучим логистическую регрессию с balanced-классами
model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)
model.fit(X_train_tfidf, y_train)

# Сохраним модель
joblib.dump(model, '/content/logreg_balanced.pkl')

In [ ]:
# 7. Валидация без GridSearch (просто предсказание на валидационной выборке)
y_val_pred = model.predict(X_val_tfidf)
print("=== Classification Report (валидация) ===")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=['human', 'AI']
))
print("=== Confusion Matrix (валидация) ===")
cm_val = confusion_matrix(y_val, y_val_pred, labels=['human', 'AI'])
disp_val = ConfusionMatrixDisplay(confusion_matrix=cm_val, display_labels=['human', 'AI'])
disp_val.plot(cmap='Blues', values_format='d')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Confusion Matrix — Validation')
plt.show()

# 8. Тестирование на тестовой выборке
y_test_pred = model.predict(X_test_tfidf)

# 8.1. Матрица ошибок (тест)
cm = confusion_matrix(y_test, y_test_pred, labels=['human', 'AI'])
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix (Test):")
print(cm)
print(f"True Negatives (human→human): {tn}")
print(f"False Positives (human→AI):   {fp}")
print(f"False Negatives (AI→human):    {fn}")
print(f"True Positives (AI→AI):        {tp}\n")

print("Classification Report (Test):")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=['human', 'AI']
))

# 8.2. Метрики для класса “AI”
prec  = precision_score(y_test, y_test_pred, pos_label='AI')
rec   = recall_score(   y_test, y_test_pred, pos_label='AI')
f1    = f1_score(       y_test, y_test_pred, pos_label='AI')
print(f"AI class — Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}\n")

# 8.3. Эксперимент с порогом probabilities
probs = model.predict_proba(X_test_tfidf)[:, 1]  # вероятности для "AI"
threshold = 0.6  # повысим порог для большей точности
y_test_thresh = np.where(probs >= threshold, 'AI', 'human')
prec_t = precision_score(y_test, y_test_thresh, pos_label='AI')
rec_t  = recall_score(   y_test, y_test_thresh, pos_label='AI')
f1_t   = f1_score(       y_test, y_test_thresh, pos_label='AI')
print(f"Threshold {threshold} — Precision: {prec_t:.3f}, Recall: {rec_t:.3f}, F1: {f1_t:.3f}\n")

In [ ]:
# 9. Функции для визуализации метрик и матриц
def plot_conf_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=['human', 'AI'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['human', 'AI'])
    disp.plot(cmap='Blues', values_format='d')
    plt.xlabel('Предсказанный класс')
    plt.ylabel('Истинный класс')
    plt.title(title)
    plt.show()

def plot_metrics_set(X, y_true, name, model):
    y_pred_local = model.predict(X)
    prec_local  = precision_score(y_true, y_pred_local, pos_label='AI')
    rec_local   = recall_score(y_true, y_pred_local, pos_label='AI')
    f1_local    = f1_score(y_true, y_pred_local, pos_label='AI')
    print(f"{name} — Precision: {prec_local:.3f}, Recall: {rec_local:.3f}, F1: {f1_local:.3f}")
    plot_conf_matrix(y_true, y_pred_local, f"Матрица ошибок")

In [ ]:
# Визуализация метрик на train/val/test
plot_metrics_set(X_train_tfidf, y_train, "Train",      model)
plot_metrics_set(X_val_tfidf,   y_val,   "Validation", model)
plot_metrics_set(X_test_tfidf,  y_test,  "Test",       model)

In [ ]:
# Предполагается, что:
# - svm_pipeline — обученный Pipeline с LinearSVC
# - X_test — массив текстов для тестирования
# - y_test — серии/массив меток строкового типа: 'human' или 'AI'

# 1. Получаем decision scores от SVM
y_scores = svm_pipeline.decision_function(X_test)  # shape = (n_samples,)

# 2. Перекодируем y_test в бинарный формат 0/1 (0 = human, 1 = AI)
mapping = {svm_pipeline.classes_[0]: 0, svm_pipeline.classes_[1]: 1}
y_test_bin = y_test.map(mapping)

# 3. Считаем ROC‐метрики (FPR, TPR и пороги) и AUC
fpr, tpr, thresholds = roc_curve(y_test_bin, y_scores)
roc_auc = auc(fpr, tpr)

# 4. Рисуем ROC‐кривую
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)  # диагональная линия случайного классификатора
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC‐кривая (SVM), класс «AI» = положительный')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# Предполагается, что у вас уже есть:
# - X_train, y_train: обучающая выборка
# - X_test,  y_test:  тестовая выборка (y_test — numpy.ndarray меток 'human' и 'AI')

# 1. Строим пайплайн с TF-IDF и LogisticRegression
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        C=1.0,
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

# 2. Обучаем LogisticRegression
lr_pipeline.fit(X_train, y_train)

# 3. Получаем вероятности для «положительного» класса 'AI'
y_prob_pos = lr_pipeline.predict_proba(X_test)[:, 1]  # столбец с вероятностью AI

# 4. Перекодируем y_test (numpy.ndarray) в бинарный формат 0/1 (0 = human, 1 = AI)
mapping = {lr_pipeline.classes_[0]: 0, lr_pipeline.classes_[1]: 1}
y_test_bin = np.array([mapping[label] for label in y_test])

# 5. Считаем ROC‐метрики и AUC
fpr, tpr, thresholds = roc_curve(y_test_bin, y_prob_pos)
roc_auc = auc(fpr, tpr)

# 6. Рисуем ROC‐кривую
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)  # диагональная линия случайного классификатора
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC‐кривая (Logistic Regression)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# 12. Визуализация TF-IDF → SVD → t-SNE
# 12.1. Берём TF-IDF из обученного векторизатора
tfidf = vectorizer
X_all_tfidf = tfidf.transform(df['clean_text'])

# 12.2. Снижаем размерность через TruncatedSVD до 50 компонент
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_all_tfidf)

# 12.3. Применяем t-SNE (2D)
tsne = TSNE(
    n_components=2,
    random_state=42,
    init='pca',
    learning_rate='auto'
)
X_tsne = tsne.fit_transform(X_svd)

# 12.4. Строим scatter plot, раскрашивая по меткам «human» и «AI»
plt.figure(figsize=(8, 6))
for label in ['human', 'AI']:
    idx = (df['label'] == label)
    plt.scatter(
        X_tsne[idx, 0],
        X_tsne[idx, 1],
        label=label,
        alpha=0.6,
        s=20
    )

plt.xlabel('t-SNE компонента 1')
plt.ylabel('t-SNE компонента 2')
plt.title('t-SNE проекция TF-IDF признаков (Logistic Regression)')
plt.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
